In [15]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Create Outlook CDI NetCDF with data only up to 2018 and generate summary.txt

This script calculates outlook CDI values based on the consistency of drought status
between consecutive months, saves them to a NetCDF file, and creates a summary.txt file
with details about the calculation and results.
"""

import os
import sys
import logging
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("outlook_cdi.log")
    ]
)
logger = logging.getLogger(__name__)

def load_original_cdi(file_path, end_year=2018):
    """
    Load the original CDI file and extract data up to the specified end year.
    
    Args:
        file_path (str): Path to the original CDI file
        end_year (int): End year for the data (inclusive)
        
    Returns:
        xarray.Dataset: The loaded CDI dataset filtered to the specified end year
    """
    logger.info(f"Step 1: Loading original CDI file from {file_path}")
    
    ds = xr.open_dataset(file_path)
    
    # Log basic information
    logger.info(f"  Original dimensions: {ds.dims}")
    logger.info(f"  Variables: {list(ds.variables)}")
    
    # Convert time values to datetime for easier filtering
    times = pd.to_datetime(ds.time.values)
    logger.info(f"  Original time range: {times[0]} to {times[-1]}")
    logger.info(f"  Total time steps: {len(times)}")
    
    # Filter to include only data up to the specified end year
    time_mask = times.year <= end_year
    filtered_times = times[time_mask]
    
    logger.info(f"  Filtering to data up to {end_year}")
    logger.info(f"  Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    logger.info(f"  Filtered time steps: {len(filtered_times)}")
    
    # Create a new dataset with only the filtered time steps
    ds_filtered = ds.sel(time=ds.time[time_mask])
    
    return ds_filtered, filtered_times

def calculate_outlook_cdi(ds, drought_threshold=0.3):
    """
    Calculate outlook CDI values based on consistency between consecutive months.
    
    Args:
        ds (xarray.Dataset): Original CDI dataset (already filtered to desired time range)
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        tuple: (ds_outlook, stats_dict) - Output dataset and statistics dictionary
    """
    logger.info(f"Step 2: Calculating outlook CDI values (threshold={drought_threshold})")
    
    # Extract dimensions and coordinates
    height = ds.dims['latitude']
    width = ds.dims['longitude']
    time_steps = ds.dims['time']
    
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    times = ds.time.values
    times_dt = pd.to_datetime(times)
    
    logger.info(f"  Processing {time_steps} time steps from {times_dt[0]} to {times_dt[-1]}")
    
    # Initialize array for outlook CDI values
    outlook_cdi = np.full((height, width, time_steps), np.nan, dtype=np.float32)
    
    # Track statistics for summary
    stats = {
        'monthly_stats': [],
        'total_cells_processed': 0,
        'total_consistent': 0,
        'total_inconsistent': 0,
        'start_time': times_dt[0],
        'end_time': times_dt[-1],
        'total_time_steps': time_steps,
        'processed_time_steps': time_steps - 1,  # Last one can't be processed
    }
    
    # Process each time step (except the last one)
    processed_count = 0
    for i in range(time_steps - 1):
        current_time = times_dt[i]
        next_time = times_dt[i + 1]
        
        # Log every 10th step to avoid excessive output
        if i % 10 == 0 or i == time_steps - 2:
            logger.info(f"  Processing time step {i+1}/{time_steps-1}: {current_time.strftime('%Y-%m')} -> {next_time.strftime('%Y-%m')}")
        
        # Get CDI values
        current_cdi = ds.cdi.values[:, :, i]
        next_cdi = ds.cdi.values[:, :, i + 1]
        
        # Determine drought status
        current_drought = (current_cdi <= drought_threshold)
        next_drought = (next_cdi <= drought_threshold)
        
        # Calculate consistency (1 if same status, 0 if different)
        consistency = (current_drought == next_drought).astype(np.float32)
        
        # Apply mask for valid data
        valid_mask = ~np.isnan(current_cdi) & ~np.isnan(next_cdi)
        consistency_pct = consistency * 100.0  # Convert to percentage
        consistency_pct[~valid_mask] = np.nan
        
        # Store at the current time index
        outlook_cdi[:, :, i] = consistency_pct
        processed_count += 1
        
        # Collect statistics for this month
        valid_values = consistency_pct[valid_mask]
        if valid_values.size > 0:
            consistent_count = np.sum(valid_values == 100.0)
            inconsistent_count = np.sum(valid_values == 0.0)
            total_valid = valid_values.size
            consistent_pct = consistent_count / total_valid * 100
            
            # Store monthly stats
            month_stats = {
                'month': current_time.strftime('%Y-%m'),
                'next_month': next_time.strftime('%Y-%m'),
                'valid_cells': total_valid,
                'consistent_cells': consistent_count,
                'inconsistent_cells': inconsistent_count,
                'consistent_percent': consistent_pct
            }
            stats['monthly_stats'].append(month_stats)
            
            # Update totals
            stats['total_cells_processed'] += total_valid
            stats['total_consistent'] += consistent_count
            stats['total_inconsistent'] += inconsistent_count
            
            if i % 10 == 0 or i == time_steps - 2:
                logger.info(f"    Consistent: {consistent_count}/{total_valid} ({consistent_pct:.1f}%)")
                logger.info(f"    Inconsistent: {inconsistent_count}/{total_valid} ({100-consistent_pct:.1f}%)")
    
    logger.info(f"  Processed {processed_count} time steps")
    logger.info(f"  Note: The last time step ({times_dt[-1].strftime('%Y-%m')}) has no outlook (no next month to compare)")
    
    # Create a new dataset with the same structure as the original
    ds_outlook = xr.Dataset(
        coords={
            'latitude': (['latitude'], latitude),
            'longitude': (['longitude'], longitude),
            'time': (['time'], times)
        },
        attrs={
            'description': 'Drought outlook CDI based on month-to-month consistency',
            'created_date': datetime.now().strftime('%Y-%m-%d'),
            'drought_threshold': str(drought_threshold),
            'calculation_method': 'Consistency percentage between current and next month',
            'time_range': f'{times_dt[0].strftime("%Y-%m")} to {times_dt[-1].strftime("%Y-%m")}'
        }
    )
    
    # Add the outlook CDI variable
    ds_outlook['outlook_cdi'] = xr.DataArray(
        data=outlook_cdi,
        dims=['latitude', 'longitude', 'time'],
        attrs={
            'long_name': 'Drought outlook CDI',
            'units': '%',
            'valid_min': 0.0,
            'valid_max': 100.0,
            'description': 'Percentage consistency of drought status with the next month (100% = same status, 0% = different status)'
        }
    )
    
    # Calculate overall statistics
    if stats['total_cells_processed'] > 0:
        stats['overall_consistent_percent'] = stats['total_consistent'] / stats['total_cells_processed'] * 100
    else:
        stats['overall_consistent_percent'] = 0
        
    return ds_outlook, stats

def save_outlook_cdi(ds_outlook, output_file):
    """
    Save the outlook CDI dataset to a NetCDF file.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        output_file (str): Path to save the output file
        
    Returns:
        str: Path to the saved file
    """
    logger.info(f"Step 3: Saving outlook CDI to NetCDF file: {output_file}")
    
    # Make sure the directory exists if needed
    directory = os.path.dirname(output_file)
    if directory:
        os.makedirs(directory, exist_ok=True)
    
    # Save the dataset
    ds_outlook.to_netcdf(output_file)
    
    # Get file size
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    
    logger.info(f"  File saved successfully: {output_file} ({file_size_mb:.1f} MB)")
    logger.info(f"  Variables: {list(ds_outlook.data_vars)}")
    logger.info(f"  Time steps: {ds_outlook.dims['time']}")
    logger.info(f"  This file has the same structure as the original CDI file")
    logger.info(f"  You can view it by time step to see the outlook values")
    
    return output_file

def create_summary_file(stats, original_file, output_file, drought_threshold, summary_file="summary.txt"):
    """
    Create a summary text file with details about the calculation and results.
    
    Args:
        stats (dict): Statistics dictionary
        original_file (str): Path to the original CDI file
        output_file (str): Path to the output CDI file
        drought_threshold (float): Threshold used for drought classification
        summary_file (str): Path to save the summary file
        
    Returns:
        str: Path to the summary file
    """
    logger.info(f"Step 4: Creating summary file: {summary_file}")
    
    # Format a nicely formatted date range
    start_date = stats['start_time'].strftime('%B %Y')
    end_date = stats['end_time'].strftime('%B %Y')
    
    # Calculate the overall consistent percentage
    overall_consistent_pct = stats['overall_consistent_percent']
    
    with open(summary_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("DROUGHT OUTLOOK CDI CALCULATION SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("CALCULATION DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Date processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Original CDI file: {original_file}\n")
        f.write(f"Output file: {output_file}\n")
        f.write(f"Drought threshold: {drought_threshold}\n")
        f.write(f"Time range: {start_date} to {end_date}\n")
        f.write(f"Total time steps in output: {stats['total_time_steps']}\n")
        f.write(f"Time steps processed: {stats['processed_time_steps']}\n\n")
        
        f.write("CALCULATION METHOD:\n")
        f.write("-" * 50 + "\n")
        f.write("Outlook CDI represents the consistency of drought status between consecutive months:\n")
        f.write(f"  - Drought defined as: CDI ≤ {drought_threshold}\n")
        f.write("  - For each grid cell, consecutive months are compared:\n")
        f.write("    * 100% = Both months have the same status (both drought or both non-drought)\n")
        f.write("    * 0% = Months have different status (one drought, one non-drought)\n")
        f.write("    * NaN = No data available or last month (no next month to compare)\n\n")
        
        f.write("OVERALL STATISTICS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Total grid cells processed: {stats['total_cells_processed']:,}\n")
        f.write(f"Consistent cells (100%): {stats['total_consistent']:,} ({stats['overall_consistent_percent']:.1f}%)\n")
        f.write(f"Inconsistent cells (0%): {stats['total_inconsistent']:,} ({100-stats['overall_consistent_percent']:.1f}%)\n\n")
        
        f.write("MONTHLY HIGHLIGHTS:\n")
        f.write("-" * 50 + "\n")
        
        # Sort monthly stats by consistency percentage (highest first)
        sorted_months = sorted(stats['monthly_stats'], key=lambda x: x['consistent_percent'], reverse=True)
        
        # Find months with highest and lowest consistency
        if sorted_months:
            most_consistent = sorted_months[0]
            least_consistent = sorted_months[-1]
            
            f.write("Month with highest consistency:\n")
            f.write(f"  {most_consistent['month']} → {most_consistent['next_month']}: {most_consistent['consistent_percent']:.1f}% consistent\n\n")
            
            f.write("Month with lowest consistency:\n")
            f.write(f"  {least_consistent['month']} → {least_consistent['next_month']}: {least_consistent['consistent_percent']:.1f}% consistent\n\n")
        
        f.write("USAGE NOTES:\n")
        f.write("-" * 50 + "\n")
        f.write("The output file has the same structure as the original CDI file:\n")
        f.write("  - Open it in your visualization tool\n")
        f.write("  - Select any time step to view the outlook values\n")
        f.write("  - Values indicate consistency with next month's drought status\n")
        f.write("  - High values (100%) indicate areas where drought status is likely to stay the same\n")
        f.write("  - Low values (0%) indicate areas where drought status is likely to change\n\n")
        
        f.write("=" * 80 + "\n")
        f.write("End of Summary\n")
        f.write("=" * 80 + "\n")
    
    logger.info(f"  Summary file created: {summary_file}")
    return summary_file

def main():
    """Main function to run the complete process."""
    start_time = datetime.now()
    logger.info("=" * 80)
    logger.info("CREATE OUTLOOK CDI NETCDF (ONLY UP TO 2018, NO ORIGINAL CDI)")
    logger.info("=" * 80)
    
    # Configuration
    config = {
        'cdi_file': "/Volumes/data/nacp/results/netcdf/cdi_1.nc",  # Original CDI file
        'output_file': "outlook_cdi_2018.nc",  # Output file
        'summary_file': "summary.txt",  # Summary file
        'drought_threshold': 0.3,  # Threshold for drought classification
        'end_year': 2018   # End year for analysis
    }
    
    # Step 1: Load original CDI file and filter to end year
    ds_filtered, filtered_times = load_original_cdi(config['cdi_file'], config['end_year'])
    
    # Step 2: Calculate outlook CDI values
    ds_outlook, stats = calculate_outlook_cdi(
        ds_filtered,
        drought_threshold=config['drought_threshold']
    )
    
    # Step 3: Save outlook CDI to NetCDF file
    output_file = save_outlook_cdi(ds_outlook, config['output_file'])
    
    # Step 4: Create summary file
    summary_file = create_summary_file(
        stats,
        config['cdi_file'],
        output_file,
        config['drought_threshold'],
        config['summary_file']
    )
    
    # Log completion
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    logger.info("=" * 80)
    logger.info(f"PROCESS COMPLETED IN {duration:.1f} SECONDS")
    logger.info("=" * 80)
    logger.info(f"Output file: {output_file}")
    logger.info(f"Summary file: {summary_file}")
    logger.info("=" * 80)

if __name__ == "__main__":
    main()

2025-05-21 10:27:16 [INFO] ================================================================================
2025-05-21 10:27:16 [INFO] CREATE OUTLOOK CDI NETCDF (ONLY UP TO 2018, NO ORIGINAL CDI)
2025-05-21 10:27:16 [INFO] ================================================================================
2025-05-21 10:27:16 [INFO] Step 1: Loading original CDI file from /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-05-21 10:27:16 [INFO]   Original dimensions: Frozen({'latitude': 681, 'longitude': 841, 'time': 319})
2025-05-21 10:27:16 [INFO]   Variables: ['latitude', 'longitude', 'time', 'cdi']
2025-05-21 10:27:16 [INFO]   Original time range: 1998-04-01 00:00:00 to 2025-04-01 00:00:00
2025-05-21 10:27:16 [INFO]   Total time steps: 319
2025-05-21 10:27:16 [INFO]   Filtering to data up to 2018
2025-05-21 10:27:16 [INFO]   Filtered time range: 1998-04-01 00:00:00 to 2018-12-01 00:00:00
2025-05-21 10:27:16 [INFO]   Filtered time steps: 249
2025-05-21 10:27:16 [INFO] Step 2: Calculating out